# Explore the Workspace and Compute

Read the configured Azure ML workspace and compute, then inspect representative models, environments, data assets, endpoints, jobs, and pipelines. This notebook performs no mutation.

**Source:** Adapted from the local Azure ML SDK v2 notebook patterns documented in [source attribution](../../../workshop/SOURCES.md).

In [ ]:
from pathlib import Path
from azure.ai.ml import MLClient
from azure.identity import AzureCliCredential
from dotenv import dotenv_values
import requests
from azure.core.credentials import TokenCredential
from azure.ai.ml.entities import AmlCompute
from itertools import islice
import pandas as pd

notebook_file = globals().get("__vsc_ipynb_file__")
search_start = Path(notebook_file).resolve().parent if notebook_file else Path.cwd().resolve()
for parent in (search_start, *search_start.parents):
    candidate = parent if parent.name == "workspace_readiness" else parent / "notebooks/workspace_readiness"
    if (candidate / ".env.example").is_file():
        SUITE_ROOT = candidate
        break
else:
    raise FileNotFoundError("Run from this repository or notebooks/workspace_readiness.")

SAMPLE_ROOT = SUITE_ROOT.parents[1] / "workshop"

env_path = SUITE_ROOT / ".env"
if not env_path.is_file():
    raise FileNotFoundError(f"Configure {env_path} with your four Azure resource values.")
env = dotenv_values(env_path, encoding="utf-8-sig", interpolate=False)
SUBSCRIPTION_ID = (env["AZURE_SUBSCRIPTION_ID"] or "").strip()
RESOURCE_GROUP = (env["AZURE_RESOURCE_GROUP"] or "").strip()
WORKSPACE_NAME = (env["AZUREML_WORKSPACE_NAME"] or "").strip()
REGISTRY_NAME = (env["AZUREML_REGISTRY_NAME"] or "").strip()

if len(env) != 4 or not all((SUBSCRIPTION_ID, RESOURCE_GROUP, WORKSPACE_NAME, REGISTRY_NAME)):
    raise ValueError("Fill in only the four workspace and registry values in .env.")
if any(value.startswith(("<", "${")) for value in (SUBSCRIPTION_ID, RESOURCE_GROUP, WORKSPACE_NAME, REGISTRY_NAME)):
    raise ValueError("Replace the .env placeholders with your Azure resource values.")

credential = AzureCliCredential(process_timeout=60)
ml_client = MLClient(credential, SUBSCRIPTION_ID, RESOURCE_GROUP, WORKSPACE_NAME)
print(f"Workspace: {WORKSPACE_NAME}; registry: {REGISTRY_NAME}")

ASSET_NAMES = {
    "data": "readiness-taxi-data",
    "environment": "readiness-taxi-environment",
    "model": "readiness-taxi-model",
}

def arm_get(credential: TokenCredential, resource_id: str, api_version: str) -> dict:
    response = requests.get(
        f"https://management.azure.com{resource_id}",
        params={"api-version": api_version},
        headers={"Authorization": f"Bearer {credential.get_token('https://management.azure.com/.default').token}"},
        timeout=60,
    )
    response.raise_for_status()
    document = response.json()
    if not isinstance(document, dict):
        raise ValueError(f"Expected an ARM resource object for {resource_id}.")
    return document

def resolve_cluster(
    client: MLClient,
    credential: TokenCredential,
    compute_name: str | None = None,
) -> tuple[str, str]:
    if compute_name:
        compute = client.compute.get(compute_name)
    else:
        clusters = [item for item in client.compute.list() if isinstance(item, AmlCompute)]
        if len(clusters) != 1:
            raise ValueError(
                f"Expected one compute cluster, found {len(clusters)}. "
                "Set COMPUTE_NAME explicitly in this notebook's configuration cell."
            )
        name = clusters[0].name
        if not name:
            raise ValueError("The compute cluster has no name.")
        compute = client.compute.get(name)

    if not isinstance(compute, AmlCompute) or not compute.name:
        raise ValueError("COMPUTE_NAME must select a named AML compute cluster.")
    identities = compute.identity.user_assigned_identities if compute.identity else []
    if not identities or len(identities) != 1:
        raise ValueError(f"Cluster {compute.name} must have exactly one user-assigned managed identity.")
    identity = identities[0]
    client_id = identity.client_id
    if not client_id and identity.resource_id:
        resource = arm_get(credential, identity.resource_id, "2023-01-31")
        client_id = resource.get("properties", {}).get("clientId")
    if not isinstance(client_id, str) or not client_id:
        raise ValueError(f"Unable to resolve the client ID for cluster {compute.name}'s identity.")
    return compute.name, client_id


COMPUTE_NAME = None  # Set a name here only when the workspace has multiple clusters.
COMPUTE_NAME, COMPUTE_IDENTITY_CLIENT_ID = resolve_cluster(ml_client, credential, COMPUTE_NAME)
OUTPUT_DATASTORE = ml_client.datastores.get_default().name


In [ ]:
workspace = ml_client.workspaces.get(WORKSPACE_NAME)
compute = ml_client.compute.get(COMPUTE_NAME)

print(f"Workspace: {workspace.name}")
print(f"Location: {workspace.location}")
print(f"Compute: {compute.name}")
print(f"Compute type: {compute.type}")
print(f"Compute state: {getattr(compute, 'state', 'not reported')}")

collections = {
    "data": ml_client.data.list(),
    "model": ml_client.models.list(),
    "environment": ml_client.environments.list(),
    "endpoint": ml_client.online_endpoints.list(),
    "job": ml_client.jobs.list(),
}
rows = [
    {"asset_type": asset_type, "name": item.name}
    for asset_type, items in collections.items()
    for item in islice(items, 5)
]
asset_inventory = pd.DataFrame(rows)
display(asset_inventory)

assert workspace.name == WORKSPACE_NAME
assert compute.name == COMPUTE_NAME
print("Read-only workspace tour complete.")

## Validate Managed Networking

Read the current ARM configuration; do not create or approve endpoints.

In [ ]:
metadata = arm_get(credential, workspace.id, "2025-09-01")
network = metadata["properties"]["managedNetwork"]
print(f"Managed-network isolation: {network['isolationMode']}")
print(f"Managed-network state: {network['status']['status']}")
if network["status"]["status"] != "Active":
    raise RuntimeError("The managed network is not Active; finish provisioning before job validation.")
registry_rules = [
    {"name": name, "status": rule["status"], "target": rule["destination"]["serviceResourceId"]}
    for name, rule in network["outboundRules"].items()
    if rule["type"] == "PrivateEndpoint"
    and rule["destination"]["serviceResourceId"].lower().endswith(
        f"/registries/{REGISTRY_NAME}".lower()
    )
]
display(pd.DataFrame(registry_rules))
if not registry_rules or any(rule["status"] != "Active" for rule in registry_rules):
    raise RuntimeError("No Active managed private-endpoint rule targets the configured registry.")
storage = arm_get(credential, metadata["properties"]["storageAccount"], "2023-05-01")
print(f"Workspace Storage public access: {storage['properties']['publicNetworkAccess']}")
if storage["properties"]["publicNetworkAccess"] == "Disabled":
    print("Storage is private. Run upload notebooks from privately connected compute, not the public laptop path.")
print("Active endpoints establish network provisioning, not registry asset RBAC or successful job traffic.")


## Expected Result

The configured workspace and compute are available, and a small cross-section of predeployed assets is visible. No resources are changed.

Next: `02_register_data_asset.ipynb`.